## Solid-solid transformation 

$\gamma$-$\gamma'$ rafting in nickel-based superalloy is a typy of spinodal decomposition under the influence of strain energy. Nickel-based superalloys are used in turbine blades, which are inch-sized single grain alloys.

<div align="left">
<img src="https://www.phase-trans.msm.cam.ac.uk/2003/Superalloys/Desktop-Images/0.jpg" width="300">
</div>

*image source:https://www.phase-trans.msm.cam.ac.uk/2003/Superalloys/Desktop-Images/0.jpg*

--- 
The cell below sets up the simulation parameters and containers.


In [ ]:
import numpy as np
import matplotlib.pyplot as plt

# -----------------------------
# discretization
# -----------------------------
Ny = 128
Nx = 128
Nstp = 10001

# -----------------------------
# simulation parameters
# -----------------------------
Omg = 2.9
kap = 1.5
dx = 1.0
dt = 1.0e-2
Mob = 1.0

# -----------------------------
# elastic constants
# -----------------------------
# Case 1
C11 = 157.9 * 20
C12 = 113.7 * 20
C44 = 70.5  * 20

# Case 2 
# C11 = 347.6 * 10
# C12 = 121.8 * 10
# C44 = 81.5  * 10

# Case 3
# C11 = 250 * 8
# C12 = 50 * 8
# C44 = 100  * 8

# anisotropic factor
aniso_factor = 2.0 * C44 / (C11 - C12)
print("anisotropic factor =", aniso_factor)

# -----------------------------
# arrays
# -----------------------------
phi   = np.zeros((Ny, Nx))
df_dp = np.zeros((Ny, Nx))
Lap   = np.zeros((Ny, Nx))
mu    = np.zeros((Ny, Nx))

u = np.zeros((Ny, Nx))
v = np.zeros((Ny, Nx))

eps0_xx = 0.02
eps0_yy = 0.02

eps_xx = np.zeros((Ny, Nx))
eps_yy = np.zeros((Ny, Nx))

b_x = np.zeros((Ny, Nx))
b_y = np.zeros((Ny, Nx))

RHS_x = np.zeros((Ny, Nx))
RHS_y = np.zeros((Ny, Nx))

ResU = np.zeros((Ny, Nx))
ResV = np.zeros((Ny, Nx))

dW_dp = np.zeros((Ny, Nx))

# -----------------------------
# initial condition with periodic BC
# -----------------------------
phi[1:-1, 1:-1] = np.random.rand(Ny - 2, Nx - 2) * 0.2 + 0.4

phi[0, :]  = phi[Ny - 2, :]
phi[-1, :] = phi[1, :]
phi[:, 0]  = phi[:, Nx - 2]
phi[:, -1] = phi[:, 1]

# -----------------------------
# plotting setup (surf(view(2)) -> imshow)
# -----------------------------
plt.ion()
fig, ax = plt.subplots(figsize=(5, 4))
im = ax.imshow(phi, origin='lower', vmin=0.0, vmax=1.0)
ax.set_title("phi")
plt.colorbar(im, ax=ax)

---
## linear elasticity solver

### Functionalized coding

Instead of a linear sequential code, standard software engineering requires functionalize and object-oriented coding. 

The cell below builds a solver for 2D linear elasticity problems. It uses an iterative method to solve for $u$ and $v$ in the 2D mechanical equilibrium equations:

$$C_{11} \frac{\partial^2 u}{\partial x^2} + C_{44} \frac{\partial^2 u}{\partial y^2} + \big( C_{12} + C_{44} \big) \frac{\partial^2 v}{\partial x \partial y}= \big(C_{11}\varepsilon_{11}^0 + C_{12} \varepsilon_{22}^0 \big)\frac{\partial \phi}{\partial x},$$

$$C_{44} \frac{\partial^2 v}{\partial x^2} + C_{11} \frac{\partial^2 v}{\partial y^2} + \big( C_{12} + C_{44} \big) \frac{\partial^2 u}{\partial x \partial y}= \big(C_{12}\varepsilon_{11}^0 + C_{11} \varepsilon_{22}^0 \big)\frac{\partial \phi}{\partial y}.$$

In [ ]:
import numpy as np

def solve_elasticity(
    u, v,
    b_x, b_y,
    RHS_x, RHS_y,
    ResU, ResV,
    C11, C12, C44,
    dx,
    tol=0.01,
    max_iter=100):
    """
    Jacobi-like iterative solver for 2D elasticity with periodic BC.

    All arrays are Ny x Nx:
        u, v    : displacement fields (updated in place)
        b_x, b_y: body forces
        RHS_x,y : workspaces
        ResU,V  : residual workspaces
    """

    Ny, Nx = u.shape

    # interior and neighbor slices
    ii = slice(1, -1)
    ip = slice(2, None)   # +1
    im = slice(0, -2)     # -1

    norm_U = 1.0
    norm_V = 1.0
    cnt = 1

    dv_dxdy = ((v[ip, ip] - v[im, ip]) / (2.0 * dx)
            -  (v[ip, im] - v[im, im]) / (2.0 * dx)) / (2.0 * dx)

    du_dxdy = ((u[ip, ip] - u[im, ip]) / (2.0 * dx)
             - (u[ip, im] - u[im, im]) / (2.0 * dx)) / (2.0 * dx)    

    # --------------------
    # RHS for u
    # --------------------
    RHS_x[ii, ii] = b_x[ii, ii] - (C12 + C44) * dv_dxdy

    # --------------------
    # RHS for v
    # --------------------
    RHS_y[ii, ii] = b_y[ii, ii] - (C12 + C44) * du_dxdy

    # iterative solver for linear elastic problem
    while (norm_U > tol) or (norm_V > tol):

        # --------------------
        # update u
        # --------------------
        u[ii, ii] = (RHS_x[ii, ii] * dx**2 - C11 * (u[ii, ip] + u[ii, im])
            - C44 * (u[ip, ii] + u[im, ii])) / (-2.0 * C11 - 2.0 * C44)

        # --------------------
        # update v
        # --------------------
        v[ii, ii] = (RHS_y[ii, ii] * dx**2 - C44 * (v[ii, ip] + v[ii, im])
            - C11 * (v[ip, ii] + v[im, ii])) / (-2.0 * C44 - 2.0 * C11)

        # --------------------
        # periodic BC for u
        # --------------------
        u[0, :]  = u[Ny - 2, :]
        u[-1, :] = u[1, :]
        u[:, 0]  = u[:, Nx - 2]
        u[:, -1] = u[:, 1]

        # periodic BC for v
        v[0, :]  = v[Ny - 2, :]
        v[-1, :] = v[1, :]
        v[:, 0]  = v[:, Nx - 2]
        v[:, -1] = v[:, 1]

        dv_dxdy = ((v[ip, ip] - v[im, ip]) / (2.0 * dx)
                -  (v[ip, im] - v[im, im]) / (2.0 * dx)) / (2.0 * dx)

        du_dxdy = ((u[ip, ip] - u[im, ip]) / (2.0 * dx)
                 - (u[ip, im] - u[im, im]) / (2.0 * dx)) / (2.0 * dx)
        
        # --------------------
        # RHS for u
        # --------------------
        RHS_x[ii, ii] = b_x[ii, ii] - (C12 + C44) * dv_dxdy

        # --------------------
        # RHS for v
        # --------------------
        RHS_y[ii, ii] = b_y[ii, ii] - (C12 + C44) * du_dxdy
        
        # --------------------
        # residuals
        # --------------------
        ResU[ii, ii] = (C11 * (u[ii, ip] - 2.0 * u[ii, ii] + u[ii, im]) / dx**2
            + C44 * (u[ip, ii] - 2.0 * u[ii, ii] + u[im, ii]) / dx**2
            - RHS_x[ii, ii])

        ResV[ii, ii] = (C44 * (v[ii, ip] - 2.0 * v[ii, ii] + v[ii, im]) / dx**2
            + C11 * (v[ip, ii] - 2.0 * v[ii, ii] + v[im, ii]) / dx**2
            - RHS_y[ii, ii])

        norm_U = np.linalg.norm(ResU[ii, ii], 2) / ((Ny - 2) * (Nx - 2))
        norm_V = np.linalg.norm(ResV[ii, ii], 2) / ((Ny - 2) * (Nx - 2))

        if cnt > max_iter:
            break
        cnt += 1

    return u, v


The cells below calculates Laplace and imposes periodic boundary conditions of a 2D array.

In [ ]:
def Laplace2d(F,dx):
    '''
    2D Laplace calculation using 5-point stencil.
    '''
    
    LapF = (F[2:, 1:-1] + F[0:-2, 1:-1] + F[1:-1, 2:] + F[1:-1, 0:-2]
        - 4.0 * F[1:-1, 1:-1]) / dx**2

    return LapF

In [ ]:
def PeriodicBC(F):
    '''
    This function imposes periodic BC on a 2d grid system.
    '''
    
    F[0, :]  = F[-2, :]
    F[-1, :] = F[1, :]
    F[:, 0]  = F[:, -2]
    F[:, -1] = F[:, 1]

    return F

---
### Evolution

The cell below implements the rafting simulaiton.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from IPython.display import display, clear_output
import time

# visualization
vmin, vmax = -0., 1.   # set to what you want

fig, ax = plt.subplots(figsize=(5,5))
im = ax.imshow(phi, origin='upper', interpolation='nearest',
               vmin=vmin, vmax=vmax, cmap='viridis')   # <- caxis via vmin/vmax
cbar = fig.colorbar(im, ax=ax, label='phi')            # <- colorbar once
ax.set_aspect('equal', adjustable='box')
ax.set_xlim(0, Nx)
ax.set_ylim(0, Ny)


# -----------------------------
# time evolution
# -----------------------------
for it in range(1, Nstp + 1):

    # --- chemical potential df/dphi ---
    phi_in = phi[1:-1, 1:-1]
    df_dp[1:-1, 1:-1] = (Omg * (1.0 - 2.0 * phi_in)+ np.log(phi_in / (1.0 - phi_in)))

    # --- Laplace of phi ---
    Lap[1:-1, 1:-1] = Laplace2d(phi[:,:],dx)

    # --- body force (misfit) in x direction ---
    b_x[1:-1, 1:-1] = ((C11 * eps0_xx + C12 * eps0_yy)
        * (phi[1:-1, 2:] - phi[1:-1, 0:-2]) / (2.0 * dx))

    # --- body force (misfit) in y direction ---
    b_y[1:-1, 1:-1] = ((C12 * eps0_xx + C11 * eps0_yy)
        * (phi[2:, 1:-1] - phi[0:-2, 1:-1]) / (2.0 * dx))

    # --- solving for displacements
    u, v = solve_elasticity(
        u, v,
        b_x, b_y,
        RHS_x, RHS_y,
        ResU, ResV,
        C11, C12, C44,
        dx,
        tol=0.01,
        max_iter=100)


    # -------------------------
    # strains
    # -------------------------
    eps_xx[1:-1, 1:-1] = (u[1:-1, 2:] - u[1:-1, 0:-2]) / (2.0 * dx)
    eps_yy[1:-1, 1:-1] = (v[2:, 1:-1] - v[0:-2, 1:-1]) / (2.0 * dx)

    # strain potential derivative dW/dphi
    eps_xx_in = eps_xx[1:-1, 1:-1]
    eps_yy_in = eps_yy[1:-1, 1:-1]

    dW_dp[1:-1, 1:-1] = (
        -(C11 * (eps_xx_in * eps0_xx + eps_yy_in * eps0_yy)
        + C12 * (eps_yy_in * eps0_xx + eps_xx_in * eps0_yy))
        + phi_in * (C11 * (eps0_xx**2 + eps0_yy**2) + 2.0 * C12 * eps0_yy * eps0_xx))

    # total chemical potential
    mu[1:-1, 1:-1] = df_dp[1:-1, 1:-1] - kap * Lap[1:-1, 1:-1] + dW_dp[1:-1, 1:-1]

    # periodic BC for mu
    mu[:,:] = PeriodicBC(mu[:,:])

    # Laplace of mu for Cahn–Hilliard equation
    Lap[1:-1, 1:-1] = Laplace2d(mu[:,:],dx)


    # time update
    phi[1:-1, 1:-1] = phi[1:-1, 1:-1] + dt * Mob * Lap[1:-1, 1:-1]

    # periodic BC for phi
    phi[:,:] = PeriodicBC(phi[:,:])

    # visualization
    if it % 100 == 1:
        im.set_data(phi)                 # update image only
        ax.set_title(f"{it}")
        
        # Animaiton part (dosn't change)
        clear_output(wait=True) # Clear output for dynamic display
        display(fig)            # Reset display
        # fig.clear()             # Prevent overlapping and layered plots
        time.sleep(0.0002)         # Sleep for half a second to slow down the animation  

